In [35]:
import pandas as pd
import sqlite3

conn = sqlite3.connect("../Dataset/olist.db")
print("Connected.")

Connected.


In [36]:
tables = ["customers", "orders", "order_items", "order_payments", 
          "order_reviews", "products", "sellers", "geolocation", "category_translation"]

for t in tables:
    df = pd.read_sql(f"SELECT * FROM {t};", conn)
    null_counts = df.isnull().sum()
    nulls_only = null_counts[null_counts > 0]
    print(f"\n--- {t} ({len(df)} rows) ---")
    if len(nulls_only) == 0:
        print("No nulls")
    else:
        print(nulls_only)


--- customers (99441 rows) ---
No nulls

--- orders (99441 rows) ---
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
dtype: int64

--- order_items (112650 rows) ---
No nulls

--- order_payments (103886 rows) ---
No nulls

--- order_reviews (99224 rows) ---
review_comment_title      87656
review_comment_message    58247
dtype: int64

--- products (32951 rows) ---
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

--- sellers (3095 rows) ---
No nulls

--- geolocation (1000163 rows) ---
No nulls

--- category_translation (71 rows) ---
No nulls


In [37]:
df = pd.read_sql("SELECT * FROM products;", conn)
print(df.isnull().sum())

product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64


In [38]:
pd.read_sql("""
    SELECT order_status, COUNT(*) as count
    FROM orders
    GROUP BY order_status
    ORDER BY count DESC;
""", conn)

,order_status,count
0,delivered,96478
1,shipped,1107
2,canceled,625
3,unavailable,609
4,invoiced,314
5,processing,301
6,created,5
7,approved,2


In [39]:
for t in tables:
    df = pd.read_sql(f"SELECT * FROM {t};", conn)
    dupes = df.duplicated().sum()
    print(f"{t}: {dupes} duplicate rows")

customers: 0 duplicate rows
orders: 0 duplicate rows
order_items: 0 duplicate rows
order_payments: 0 duplicate rows
order_reviews: 0 duplicate rows
products: 0 duplicate rows
sellers: 0 duplicate rows
geolocation: 261831 duplicate rows
category_translation: 0 duplicate rows


In [40]:
geo = pd.read_sql("SELECT * FROM geolocation;", conn)
print("Total rows:", len(geo))
print("Unique zip codes:", geo['geolocation_zip_code_prefix'].nunique())

Total rows: 1000163
Unique zip codes: 19015


In [41]:
geo_clean = geo.groupby('geolocation_zip_code_prefix').agg({
    'geolocation_lat': 'mean',
    'geolocation_lng': 'mean',
    'geolocation_city': 'first',
    'geolocation_state': 'first'
}).reset_index()

print("Cleaned geolocation rows:", len(geo_clean))
geo_clean.head()

Cleaned geolocation rows: 19015


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1001,-23.550190,-46.634024,sao paulo,SP
1,1002,-23.548146,-46.634979,sao paulo,SP
2,1003,-23.548994,-46.635731,sao paulo,SP
3,1004,-23.549799,-46.634757,sao paulo,SP
4,1005,-23.549456,-46.636733,sao paulo,SP


In [42]:
geo_clean.to_sql('geolocation_clean', conn, if_exists='replace', index=False)
print("Saved geolocation_clean table.")

Saved geolocation_clean table.


In [43]:
query = """
SELECT 
    o.order_id,
    o.customer_id,
    o.order_status,
    o.order_purchase_timestamp,
    o.order_approved_at,
    o.order_delivered_carrier_date,
    o.order_delivered_customer_date,
    o.order_estimated_delivery_date,
    oi.order_item_id,
    oi.product_id,
    oi.seller_id,
    oi.price,
    oi.freight_value
FROM orders o
JOIN order_items oi ON o.order_id = oi.order_id;
"""
order_items_full = pd.read_sql(query, conn)
print("Rows:", len(order_items_full))
order_items_full.head()

Rows: 112650


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_item_id,product_id,seller_id,price,freight_value
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,1,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,29.99,8.72
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,1,595fac2a385ac33a80bd5114aec74eb8,289cdb325fb7e7f891c38608bf9e0962,118.70,22.76
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,1,aa4383b373c6aca5d8797843e5594415,4869f7a5dfa277a7dca6462dcf3b52b2,159.90,19.22
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,1,d0b61bfb1de832b15ba9d266ca96e5b0,66922902710d126a0e7d26b0e3805106,45.00,27.20
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,1,65266b2da20d04dbe00c5c2d3bb7859e,2c9e548be18521d1c43cde1c582c6de8,19.90,8.72


In [44]:
query = """
SELECT 
    o.order_id,
    o.customer_id,
    o.order_status,
    o.order_purchase_timestamp,
    o.order_approved_at,
    o.order_delivered_carrier_date,
    o.order_delivered_customer_date,
    o.order_estimated_delivery_date,
    oi.order_item_id,
    oi.product_id,
    oi.seller_id,
    oi.price,
    oi.freight_value,
    c.customer_city,
    c.customer_state,
    s.seller_city,
    s.seller_state
FROM orders o
JOIN order_items oi ON o.order_id = oi.order_id
JOIN customers c ON o.customer_id = c.customer_id
JOIN sellers s ON oi.seller_id = s.seller_id;
"""
order_full = pd.read_sql(query, conn)
print("Rows:", len(order_full))
order_full.head()

Rows: 112650


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_item_id,product_id,seller_id,price,freight_value,customer_city,customer_state,seller_city,seller_state
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,1,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,29.99,8.72,sao paulo,SP,maua,SP
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,1,595fac2a385ac33a80bd5114aec74eb8,289cdb325fb7e7f891c38608bf9e0962,118.70,22.76,barreiras,BA,belo horizonte,SP
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,1,aa4383b373c6aca5d8797843e5594415,4869f7a5dfa277a7dca6462dcf3b52b2,159.90,19.22,vianopolis,GO,guariba,SP
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,1,d0b61bfb1de832b15ba9d266ca96e5b0,66922902710d126a0e7d26b0e3805106,45.00,27.20,sao goncalo do amarante,RN,belo horizonte,MG
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,1,65266b2da20d04dbe00c5c2d3bb7859e,2c9e548be18521d1c43cde1c582c6de8,19.90,8.72,santo andre,SP,mogi das cruzes,SP


In [45]:
query = """
SELECT 
    o.order_id,
    o.customer_id,
    o.order_status,
    o.order_purchase_timestamp,
    o.order_approved_at,
    o.order_delivered_carrier_date,
    o.order_delivered_customer_date,
    o.order_estimated_delivery_date,
    oi.order_item_id,
    oi.product_id,
    oi.seller_id,
    oi.price,
    oi.freight_value,
    c.customer_city,
    c.customer_state,
    s.seller_city,
    s.seller_state,
    p.product_category_name,
    ct.product_category_name_english,
    r.review_score
FROM orders o
JOIN order_items oi ON o.order_id = oi.order_id
JOIN customers c ON o.customer_id = c.customer_id
JOIN sellers s ON oi.seller_id = s.seller_id
LEFT JOIN products p ON oi.product_id = p.product_id
LEFT JOIN category_translation ct ON p.product_category_name = ct.product_category_name
LEFT JOIN order_reviews r ON o.order_id = r.order_id;
"""
order_master = pd.read_sql(query, conn)
print("Rows:", len(order_master))
order_master.head()

Rows: 113314


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_item_id,product_id,seller_id,price,freight_value,customer_city,customer_state,seller_city,seller_state,product_category_name,product_category_name_english,review_score
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,1,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,29.99,8.72,sao paulo,SP,maua,SP,utilidades_domesticas,housewares,4.0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,1,595fac2a385ac33a80bd5114aec74eb8,289cdb325fb7e7f891c38608bf9e0962,118.70,22.76,barreiras,BA,belo horizonte,SP,perfumaria,perfumery,4.0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,1,aa4383b373c6aca5d8797843e5594415,4869f7a5dfa277a7dca6462dcf3b52b2,159.90,19.22,vianopolis,GO,guariba,SP,automotivo,auto,5.0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,1,d0b61bfb1de832b15ba9d266ca96e5b0,66922902710d126a0e7d26b0e3805106,45.00,27.20,sao goncalo do amarante,RN,belo horizonte,MG,pet_shop,pet_shop,5.0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,1,65266b2da20d04dbe00c5c2d3bb7859e,2c9e548be18521d1c43cde1c582c6de8,19.90,8.72,santo andre,SP,mogi das cruzes,SP,papelaria,stationery,5.0


In [46]:
review_counts = pd.read_sql("""
    SELECT order_id, COUNT(*) as review_count
    FROM order_reviews
    GROUP BY order_id
    HAVING COUNT(*) > 1
    ORDER BY review_count DESC;
""", conn)
print("Orders with multiple reviews:", len(review_counts))
review_counts.head()

Orders with multiple reviews: 547


,order_id,review_count
0,df56136b8031ecd28e200bb18e6ddb2e,3
1,c88b1d1b157a9999ce368f218a407141,3
2,8e17072ec97ce29f0e1f111e598b0c85,3
3,03c939fd7fd3b38f8485a0f95798f1f6,3
4,ffaabba06c9d293a3c614e0515ddbabc,2


In [47]:
query = """
SELECT 
    o.order_id,
    o.customer_id,
    o.order_status,
    o.order_purchase_timestamp,
    o.order_approved_at,
    o.order_delivered_carrier_date,
    o.order_delivered_customer_date,
    o.order_estimated_delivery_date,
    oi.order_item_id,
    oi.product_id,
    oi.seller_id,
    oi.price,
    oi.freight_value,
    c.customer_city,
    c.customer_state,
    s.seller_city,
    s.seller_state,
    p.product_category_name,
    ct.product_category_name_english,
    r.review_score
FROM orders o
JOIN order_items oi ON o.order_id = oi.order_id
JOIN customers c ON o.customer_id = c.customer_id
JOIN sellers s ON oi.seller_id = s.seller_id
LEFT JOIN products p ON oi.product_id = p.product_id
LEFT JOIN category_translation ct ON p.product_category_name = ct.product_category_name
LEFT JOIN (
    SELECT order_id, review_score,
           ROW_NUMBER() OVER (PARTITION BY order_id ORDER BY review_creation_date DESC) as rn
    FROM order_reviews
) r ON o.order_id = r.order_id AND r.rn = 1;
"""
order_master = pd.read_sql(query, conn)
print("Rows:", len(order_master))

Rows: 112650


In [48]:
order_master.to_sql('order_master', conn, if_exists='replace', index=False)
print("Saved order_master:", len(order_master), "rows")

Saved order_master: 112650 rows


In [49]:
order_master['order_purchase_timestamp'] = pd.to_datetime(order_master['order_purchase_timestamp'])
order_master['order_delivered_customer_date'] = pd.to_datetime(order_master['order_delivered_customer_date'])
order_master['order_estimated_delivery_date'] = pd.to_datetime(order_master['order_estimated_delivery_date'])

# Delay in days: positive = late, negative = early, NaN = not yet delivered
order_master['delivery_delay_days'] = (
    order_master['order_delivered_customer_date'] - order_master['order_estimated_delivery_date']
).dt.days

order_master.to_sql('order_master', conn, if_exists='replace', index=False)
print("Updated order_master with delivery_delay_days")
order_master[['order_id', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'delivery_delay_days']].head(10)

Updated order_master with delivery_delay_days


,order_id,order_delivered_customer_date,order_estimated_delivery_date,delivery_delay_days
0,e481f51cbdc54678b7cc49136f2d6af7,2017-10-10 21:25:13,2017-10-18,-8.0
1,53cdb2fc8bc7dce0b6741e2150273451,2018-08-07 15:27:45,2018-08-13,-6.0
2,47770eb9100c2d0c44946d9cf07ec65d,2018-08-17 18:06:29,2018-09-04,-18.0
3,949d5b44dbf5de918fe9c16f97b45f8a,2017-12-02 00:28:42,2017-12-15,-13.0
4,ad21c59c0840e6cb83a9ceb5573f8159,2018-02-16 18:17:02,2018-02-26,-10.0
5,a4591c265e18cb1dcee52889e2d8acc3,2017-07-26 10:57:55,2017-08-01,-6.0
6,136cce7faa42fdb2cefd53fdc79a6098,NaT,2017-05-09,NaN
7,6514b8ad8028c9f2cc2374ded245783f,2017-05-26 12:55:51,2017-06-07,-12.0
8,76c6e866289321a7c93b82b54852dc33,2017-02-02 14:08:10,2017-03-06,-32.0
9,e69bfb5eb88e0ed6a785585b27e16dbf,2017-08-16 17:14:30,2017-08-23,-7.0


In [50]:
seller_revenue = order_master.groupby('seller_id').agg(
    total_revenue=('price', 'sum'),
    total_orders=('order_id', 'nunique'),
    avg_order_value=('price', 'mean')
).reset_index().sort_values('total_revenue', ascending=False)

print("Total sellers:", len(seller_revenue))
seller_revenue.head(10)

Total sellers: 3095


,seller_id,total_revenue,total_orders,avg_order_value
857,4869f7a5dfa277a7dca6462dcf3b52b2,229472.63,1132,198.505735
1013,53243585a1d6dc2643021fd1853d8905,222776.05,358,543.356220
881,4a3ca9315b744ce9f8e9374361493884,200472.92,1806,100.892260
3024,fa1c13f2614d7b5c4749cbc52fecda94,194042.03,585,331.129744
1535,7c67e1448b00f6e969d365cea6b010ab,187923.89,982,137.774113
1560,7e93a43ef30c4f03f38b393420bc753a,176431.87,336,518.917265
2643,da8622b14eb17ae2831f4ac5b9dab84a,160236.57,1314,103.311779
1505,7a67c85e85bb2ce8582c35f2203ad736,141745.53,1160,121.046567
192,1025f0e2d44d7041d6cf58b6550e0bfa,138968.55,915,97.316912
1824,955fee9216a65b617aa5c0531780ce60,135171.70,1287,90.174583


In [51]:
delivered = order_master[order_master['order_status'] == 'delivered'].copy()
delivered['is_late'] = delivered['delivery_delay_days'] > 0

seller_delay = delivered.groupby('seller_id').agg(
    total_delivered=('order_id', 'nunique'),
    late_deliveries=('is_late', 'sum')
).reset_index()

seller_delay['late_rate_pct'] = (seller_delay['late_deliveries'] / seller_delay['total_delivered'] * 100).round(2)

# Only look at sellers with a meaningful volume (avoid 1-order sellers skewing the rate)
seller_delay_filtered = seller_delay[seller_delay['total_delivered'] >= 10].sort_values('late_rate_pct', ascending=False)

print("Sellers with 10+ delivered orders:", len(seller_delay_filtered))
seller_delay_filtered.head(10)

Sellers with 10+ delivered orders: 1238


,seller_id,total_delivered,late_deliveries,late_rate_pct
451,2709af9587499e95e803a6498a5a56e9,25,23,92.00
2053,b1b3948701c5c72445495bd161b83a4c,14,9,64.29
2270,c37b2059d4f90d4feead554e5246565e,12,7,58.33
446,26e2c91ef821e1ff8985f408788fe35b,12,5,41.67
31,02dcd3e8e25bee036e32512bcf175493,13,5,38.46
931,5011f0d93373a4c5753adf58ca77af8d,11,4,36.36
2375,cb41bfbcbda0aea354a834ab222f9a59,11,4,36.36
1000,54965bbe3e4f07ae045b90b0b8541f52,73,26,35.62
2766,ede0c03645598cdfc63ca8237acbe73d,43,15,34.88
1543,821fb029fc6e495ca4f08a35d51e53a5,24,8,33.33


In [52]:
seller_combined = seller_revenue.merge(seller_delay, on='seller_id', how='left')
seller_combined = seller_combined[seller_combined['total_delivered'] >= 10]

# High revenue AND high late rate = the sellers that matter most to flag
risk_sellers = seller_combined.sort_values('total_revenue', ascending=False).head(20)
risk_sellers[['seller_id', 'total_revenue', 'total_orders', 'late_rate_pct']]

,seller_id,total_revenue,total_orders,late_rate_pct
0,4869f7a5dfa277a7dca6462dcf3b52b2,229472.63,1132,10.77
1,53243585a1d6dc2643021fd1853d8905,222776.05,358,3.45
2,4a3ca9315b744ce9f8e9374361493884,200472.92,1806,10.67
3,fa1c13f2614d7b5c4749cbc52fecda94,194042.03,585,9.17
4,7c67e1448b00f6e969d365cea6b010ab,187923.89,982,12.33
5,7e93a43ef30c4f03f38b393420bc753a,176431.87,336,4.70
6,da8622b14eb17ae2831f4ac5b9dab84a,160236.57,1314,7.55
7,7a67c85e85bb2ce8582c35f2203ad736,141745.53,1160,5.24
8,1025f0e2d44d7041d6cf58b6550e0bfa,138968.55,915,10.44
9,955fee9216a65b617aa5c0531780ce60,135171.70,1287,7.53


In [53]:
correlation = seller_combined[['total_orders', 'late_rate_pct']].corr()
print(correlation)

               total_orders  late_rate_pct
total_orders       1.000000       0.011462
late_rate_pct      0.011462       1.000000


In [54]:
delivered_with_reviews = delivered.dropna(subset=['review_score'])

delay_vs_review = delivered_with_reviews.groupby(
    delivered_with_reviews['delivery_delay_days'].apply(lambda x: 'Late' if x > 0 else 'On-time/Early')
)['review_score'].agg(['mean', 'count']).round(2)

print(delay_vs_review)

                     mean   count
delivery_delay_days              
Late                 2.26    7083
On-time/Early        4.21  102287


In [55]:
delivered_reviews = order_master[
    (order_master['order_status'] == 'delivered') & 
    (order_master['review_score'].notna()) &
    (order_master['delivery_delay_days'].notna())
].copy()

# Bucket delay into simple categories for a clean chart
def delay_bucket(days):
    if days <= 0:
        return 'On time / Early'
    elif days <= 7:
        return 'Late (1-7 days)'
    else:
        return 'Very Late (8+ days)'

delivered_reviews['delay_bucket'] = delivered_reviews['delivery_delay_days'].apply(delay_bucket)

review_by_delay = delivered_reviews.groupby('delay_bucket').agg(
    avg_review_score=('review_score', 'mean'),
    order_count=('order_id', 'nunique')
).reindex(['On time / Early', 'Late (1-7 days)', 'Very Late (8+ days)'])

review_by_delay['avg_review_score'] = review_by_delay['avg_review_score'].round(2)
review_by_delay

,avg_review_score,order_count
delay_bucket,,
On time / Early,4.21,89443
Late (1-7 days),2.68,3600
Very Late (8+ days),1.70,2781


In [56]:
corr = delivered_reviews[['delivery_delay_days', 'review_score']].corr()
print(corr)

                     delivery_delay_days  review_score
delivery_delay_days             1.000000     -0.229614
review_score                   -0.229614      1.000000


In [57]:
category_reviews = delivered_reviews.groupby('product_category_name_english').agg(
    avg_review_score=('review_score', 'mean'),
    order_count=('order_id', 'nunique')
).reset_index()

# Only categories with meaningful volume
category_reviews_filtered = category_reviews[category_reviews['order_count'] >= 50].sort_values('avg_review_score')

category_reviews_filtered.head(10)

,product_category_name_english,avg_review_score,order_count
57,office_furniture,3.512696,1244
34,fixed_telephony,3.757937,209
30,fashion_male_clothing,3.758065,105
4,audio,3.837989,345
47,home_confort,3.852459,390
7,bed_bath_table,3.923183,9177
40,furniture_living_room,3.934694,409
39,furniture_decor,3.954827,6260
48,home_construction,3.957770,481
19,construction_tools_safety,3.967033,158


In [58]:
office_furn = delivered_reviews[delivered_reviews['product_category_name_english'] == 'office_furniture']
print("Office furniture avg delay:", office_furn['delivery_delay_days'].mean().round(1), "days")
print("Overall avg delay:", delivered_reviews['delivery_delay_days'].mean().round(1), "days")

Office furniture avg delay: -11.9 days
Overall avg delay: -12.1 days


In [59]:
category_cost = order_master.groupby('product_category_name_english').agg(
    total_revenue=('price', 'sum'),
    total_freight=('freight_value', 'sum'),
    order_count=('order_id', 'nunique')
).reset_index()

category_cost['freight_pct_of_revenue'] = (category_cost['total_freight'] / category_cost['total_revenue'] * 100).round(2)

# Filter to meaningful volume, sort by freight burden
category_cost_filtered = category_cost[category_cost['order_count'] >= 50].sort_values('freight_pct_of_revenue', ascending=False)
category_cost_filtered.head(10)

,product_category_name_english,total_revenue,total_freight,order_count,freight_pct_of_revenue
12,christmas_supplies,8800.82,3229.30,128,36.69
62,signaling_and_security,21509.23,6507.82,140,30.26
37,food_drink,15179.48,4507.99,227,29.70
26,electronics,160246.74,46578.32,2550,29.07
40,furniture_living_room,68916.56,17968.17,422,26.07
51,kitchen_dining_laundry_garden_furniture,46328.37,11999.43,248,25.90
24,drinks,22428.70,5741.25,297,25.60
57,office_furniture,273960.70,68571.95,1273,25.03
36,food,29393.41,7271.03,450,24.74
39,furniture_decor,729762.49,172749.30,6449,23.67


In [60]:
payment_dist = order_master.merge(
    pd.read_sql("SELECT order_id, payment_type, payment_installments FROM order_payments", conn),
    on='order_id', how='left'
)

payment_summary = payment_dist['payment_type'].value_counts(normalize=True).mul(100).round(2)
print(payment_summary)

payment_type
credit_card    73.78
boleto         19.44
voucher         5.33
debit_card      1.44
Name: proportion, dtype: float64


In [61]:
payment_dist_clean = payment_dist[payment_dist['payment_installments'].notna()]
installment_analysis = payment_dist_clean.groupby('payment_installments').agg(
    avg_price=('price', 'mean'),
    order_count=('order_id', 'nunique')
).reset_index()

installment_analysis[installment_analysis['order_count'] >= 20].sort_values('payment_installments')

,payment_installments,avg_price,order_count
1,1.0,91.144611,48609
2,2.0,97.803021,12317
3,3.0,108.299014,10385
4,4.0,126.005182,7040
5,5.0,139.312704,5195
6,6.0,156.332480,3895
7,7.0,145.151045,1612
8,8.0,237.499998,4239
9,9.0,151.219174,635
10,10.0,292.106682,5261


In [62]:
state_revenue = order_master.groupby('customer_state').agg(
    total_revenue=('price', 'sum'),
    order_count=('order_id', 'nunique')
).reset_index().sort_values('total_revenue', ascending=False)

state_revenue.head(10)

,customer_state,total_revenue,order_count
25,SP,5202955.05,41375
18,RJ,1824092.67,12762
10,MG,1585308.03,11544
22,RS,750304.02,5432
17,PR,683083.76,4998
23,SC,520553.34,3612
4,BA,511349.99,3358
6,DF,302603.94,2125
8,GO,294591.95,2007
7,ES,275037.31,2025


In [63]:
state_delay = delivered.groupby('customer_state').agg(
    total_delivered=('order_id', 'nunique'),
    late_deliveries=('is_late', 'sum')
).reset_index()

state_delay['late_rate_pct'] = (state_delay['late_deliveries'] / state_delay['total_delivered'] * 100).round(2)
state_delay_filtered = state_delay[state_delay['total_delivered'] >= 50].sort_values('late_rate_pct', ascending=False)

state_delay_filtered.head(10)

,customer_state,total_delivered,late_deliveries,late_rate_pct
1,AL,397,89,22.42
9,MA,717,144,20.08
24,SE,335,61,18.21
5,CE,1279,194,15.17
16,PI,476,71,14.92
4,BA,3256,438,13.45
18,RJ,12350,1644,13.31
13,PA,946,119,12.58
14,PB,517,63,12.19
7,ES,1995,239,11.98


In [64]:
seller_reviews = delivered_reviews.groupby('seller_id').agg(
    avg_review_score=('review_score', 'mean'),
    review_count=('order_id', 'nunique')
).reset_index()

# Merge with revenue/volume data we already built
seller_risk = seller_revenue.merge(seller_reviews, on='seller_id', how='left')

# High-impact = meaningful volume AND poor average review
seller_risk_filtered = seller_risk[
    (seller_risk['total_orders'] >= 30) & 
    (seller_risk['avg_review_score'].notna())
].sort_values('avg_review_score')

seller_risk_filtered[['seller_id', 'total_revenue', 'total_orders', 'avg_review_score']].head(10)

,seller_id,total_revenue,total_orders,avg_review_score
206,1ca7077d890b907f89be8c954a02686a,13341.57,115,2.269841
53,2eb70248d66e0e3ef83659f71b244378,42628.61,202,2.809278
710,602044f2c16190c2c6e45eb35c2e21cb,3739.16,50,2.962963
370,972d0f9cf61b499a4812cf0bfa3ad3c4,8089.29,83,2.964286
341,a49928bcdf77c55c6d6e05e09a9b4ca5,8816.70,98,2.971154
181,8e6d7754bc7e0f22c96d255ebda59eba,14497.27,85,2.992248
842,2a1348e9addc1af5aaa619b1a3679d6b,2843.00,52,3.040000
621,bbad7e518d7af88a0897397ffdca1979,4462.22,69,3.048193
272,54965bbe3e4f07ae045b90b0b8541f52,10961.30,78,3.065789
289,5058e8c1e82653974541e83690655b4a,10022.82,63,3.080000


In [65]:
# Save all analysis tables for Power BI to consume directly
seller_revenue.to_sql('agg_seller_revenue', conn, if_exists='replace', index=False)
seller_delay.to_sql('agg_seller_delay', conn, if_exists='replace', index=False)
seller_risk.to_sql('agg_seller_risk', conn, if_exists='replace', index=False)
category_cost.to_sql('agg_category_cost', conn, if_exists='replace', index=False)
review_by_delay.to_sql('agg_review_by_delay', conn, if_exists='replace', index=False)
category_reviews.to_sql('agg_category_reviews', conn, if_exists='replace', index=False)
state_revenue.to_sql('agg_state_revenue', conn, if_exists='replace', index=False)
state_delay.to_sql('agg_state_delay', conn, if_exists='replace', index=False)

print("All aggregate tables saved.")

# Confirm what's in the database now
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table';", conn)
print(tables)

All aggregate tables saved.
                    name
0              customers
1                 orders
2            order_items
3         order_payments
4          order_reviews
5               products
6                sellers
7            geolocation
8   category_translation
9      geolocation_clean
10          order_master
11    agg_seller_revenue
12      agg_seller_delay
13       agg_seller_risk
14     agg_category_cost
15   agg_review_by_delay
16  agg_category_reviews
17     agg_state_revenue
18       agg_state_delay


In [66]:
conn.close()
print("Connection closed.")

Connection closed.


In [67]:
import sqlite3
conn = sqlite3.connect("../Dataset/olist.db")

import os
export_dir = "../Dataset/powerbi_exports"
os.makedirs(export_dir, exist_ok=True)

tables_to_export = [
    "order_master",
    "agg_seller_revenue",
    "agg_seller_delay",
    "agg_seller_risk",
    "agg_category_cost",
    "agg_review_by_delay",
    "agg_category_reviews",
    "agg_state_revenue",
    "agg_state_delay",
    "geolocation_clean"
]

for t in tables_to_export:
    df = pd.read_sql(f"SELECT * FROM {t};", conn)
    df.to_csv(f"{export_dir}/{t}.csv", index=False)
    print(f"Exported {t}: {len(df)} rows")

conn.close()
print("\nDone. Files saved to Dataset/powerbi_exports/")

Exported order_master: 112650 rows
Exported agg_seller_revenue: 3095 rows
Exported agg_seller_delay: 2970 rows
Exported agg_seller_risk: 3095 rows
Exported agg_category_cost: 71 rows
Exported agg_review_by_delay: 3 rows
Exported agg_category_reviews: 71 rows
Exported agg_state_revenue: 27 rows
Exported agg_state_delay: 27 rows
Exported geolocation_clean: 19015 rows

Done. Files saved to Dataset/powerbi_exports/


In [68]:
# Monthly revenue trend
order_master['order_purchase_timestamp'] = pd.to_datetime(order_master['order_purchase_timestamp'])
order_master['order_month'] = order_master['order_purchase_timestamp'].dt.to_period('M').astype(str)

monthly_revenue = order_master.groupby('order_month').agg(
    total_revenue=('price', 'sum'),
    order_count=('order_id', 'nunique')
).reset_index().sort_values('order_month')

monthly_revenue.to_csv('../Dataset/powerbi_exports/agg_monthly_revenue.csv', index=False)
print("Monthly revenue exported:", len(monthly_revenue), "months")

# Seller combined (revenue + review + late rate, for the bubble chart)
seller_reviews_full = delivered_reviews.groupby('seller_id').agg(avg_review_score=('review_score','mean')).reset_index()
seller_bubble = seller_revenue.merge(seller_delay[['seller_id','late_rate_pct','total_delivered']], on='seller_id', how='left')
seller_bubble = seller_bubble.merge(seller_reviews_full, on='seller_id', how='left')
seller_bubble = seller_bubble[seller_bubble['total_orders'] >= 10]

seller_bubble.to_csv('../Dataset/powerbi_exports/agg_seller_bubble.csv', index=False)
print("Seller bubble data exported:", len(seller_bubble), "sellers")

Monthly revenue exported: 24 months
Seller bubble data exported: 1271 sellers
